---
# LLPS-ScanAI  
Welcome to **LLPS-ScanAI**, a user-friendly notebook to predict phase-separating driver regions in proteins using protein language models and deep learning.
- **Input**: an UniProt ID or a protein sequence.
- **Output**: LLPS propensity profile table and figure.  
- **How?** Just go to `Runtime` → `Run all` or press `ctrl+F9`  
- **Results** will be displayed at the bottom of this notebook.  

---

# 1. Set up

In [ ]:
#@title Input protein data { display-mode: "form", form-width: "100%" }

#@markdown - Enter an UniProt ID in the box below.
uniprot_id = "D0PV95"  #@param {type:"string"}
#@markdown - Or input a protein sequence directamente.
input_sequence = ""  #@param {type:"string"}

#@markdown You can try the example sequence provided below:
use_example= False  #@param {type:"boolean"}

if uniprot_id or input_sequence:
    use_example = False

if use_example:
    uniprot_id = "D0PV95"
    input_sequence = "MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA"

#@markdown ---
import time
start_time = time.time()

In [ ]:
#@title Retrieve sequence from UniProt if needed { display-mode: "form", form-width: "100%"  }
import requests
import pandas as pd

if not use_example and uniprot_id:
    uniprot_id = uniprot_id.strip()
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    if response.status_code == 200:
        fasta_lines = response.text.strip().split('\n')
        input_sequence = ''.join(fasta_lines[1:])
    else:
        raise ValueError(f"Failed to fetch sequence for UniProt ID {uniprot_id}.")

input_sequence = input_sequence.replace(' ', '').replace('\n', '').upper()
if input_sequence == "":
    raise ValueError("⚠️ No sequence provided.")
if not all(c in 'ACDEFGHIKLMNPQRSTVWY' for c in input_sequence):
    raise ValueError("⚠️ Invalid sequence provided. Please ensure the sequence contains only standard amino acid characters (A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y).")

df = pd.DataFrame({'uniprot_id': [uniprot_id], 'sequence': [input_sequence]})

In [ ]:
#@title Download model from GitLab { display-mode: "form", form-width: "100%" }
import os
import urllib.request
from tqdm import tqdm

os.makedirs("models", exist_ok=True)
base_url = "https://gitlab.com/azourbarbar/llps-scanai/-/raw/master/models"

model_files = [f"model_fold_{i}.pth" for i in range(1, 6)] + ["model_config.json"]

print("Downloading LLPS-ScanAI models...")
for fname in tqdm(model_files):
    model_url = f"{base_url}/{fname}"
    model_path = f"models/{fname}"
    if not os.path.exists(model_path):
        urllib.request.urlretrieve(model_url, model_path)

# 2. Predict

In [ ]:
%%capture
#@title Load models { display-mode: "form", form-width: "100%" }
import torch
from torch import nn
import json

# Arquitectura Conv1D (contexto local, kernel=10) + MLP -- version final entrenada
class ConvMLP(nn.Module):
    def __init__(self, in_dim=1024, conv_channels=256, kernel_size=10, p=0.6):
        super().__init__()
        self.conv = nn.Conv1d(in_dim, conv_channels, kernel_size=kernel_size, padding='same')
        self.bn_conv = nn.BatchNorm1d(conv_channels)
        self.relu_conv = nn.ReLU()
        self.dropout_conv = nn.Dropout(p)
        self.head = nn.Sequential(
            nn.Linear(conv_channels, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, 1),
        )

    def forward(self, x, mask=None):
        # x: (B, L, in_dim)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = self.bn_conv(x)
        x = self.relu_conv(x)
        x = self.dropout_conv(x)
        x = x.transpose(1, 2)
        B, L, C = x.shape
        logits_flat = self.head(x.reshape(B * L, C)).reshape(B, L)
        return logits_flat

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open('models/model_config.json', 'r') as f:
    config = json.load(f)
best_thr = config.get("best_threshold", 0.31)
kernel_size = config.get("kernel_size", 10)
conv_channels = config.get("conv_channels", 256)

llps_models = []
for i in range(1, 6):
    m = ConvMLP(in_dim=1024, conv_channels=conv_channels, kernel_size=kernel_size).to(device)
    m.load_state_dict(torch.load(f"models/model_fold_{i}.pth", map_location=device))
    m.eval()
    llps_models.append(m)

In [ ]:
#@title Generate embedding representations { display-mode: "form", form-width: "100%" }
import torch
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm

transformer_link = "Rostlab/prot_t5_xl_half_uniref50-enc"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = T5EncoderModel.from_pretrained(transformer_link).to(device).eval()
tokenizer = T5Tokenizer.from_pretrained(transformer_link, do_lower_case=False, legacy=False)

def generate_embeddings(sequence: str):
    spaced = " ".join(list(sequence))
    ids = tokenizer(spaced, add_special_tokens=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(input_ids=ids["input_ids"], attention_mask=ids["attention_mask"])
    return out.last_hidden_state[0, :-1].cpu().numpy()

tqdm.pandas(desc="Generating embeddings")
df["embedding"] = df["sequence"].progress_map(generate_embeddings)

In [ ]:
#@title Run Predictions { display-mode: "form", form-width: "100%" }
import numpy as np

def ensemble_predict(embedding):
    # embedding: (L, 1024) -> se agrega dimension de batch para la Conv1D: (1, L, 1024)
    X = torch.tensor(embedding).float().unsqueeze(0).to(device)
    mask = torch.ones(X.shape[0], X.shape[1], dtype=torch.bool).to(device)
    with torch.no_grad():
        # Soft voting real: se promedian PROBABILIDADES (sigmoid por modelo), NO logits
        all_probs = [torch.sigmoid(m(X, mask)).squeeze(0).cpu().numpy() for m in llps_models]
    return np.mean(all_probs, axis=0)

tqdm.pandas(desc="Computing probabilities")
df["prob_vector"] = df["embedding"].progress_map(ensemble_predict)

# 3. Results

In [ ]:
#@title Visualize results { display-mode: "form", form-width: "100%" }
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

window_size = 10  #@param {type:"slider", min:1, max:25, step:1}

def moving_average(x, w=5):
    serie = pd.Series(x)
    return serie.rolling(window=w, center=True, min_periods=1).mean().to_numpy()

row = df.iloc[0]
residues = list(row["sequence"])
positions = np.arange(1, len(residues) + 1)
prob_vector = np.array(row["prob_vector"])
smoothed = moving_average(prob_vector, w=window_size)

result_df = pd.DataFrame({
    "uniprot_id": [row["uniprot_id"]] * len(positions),
    "position": positions,
    "residue": residues,
    "llps_score": prob_vector
})

# Figura sola (esta es la que se exporta al descargar resultados)
fig_only = go.Figure()
fig_only.add_trace(go.Scatter(x=positions, y=prob_vector, fill='tozeroy', mode='lines', line=dict(color='lightgray'), name='Raw Score'))
fig_only.add_trace(go.Scatter(x=positions, y=smoothed, mode='lines+markers', name='Smoothed (Window)', line=dict(color='blue', width=2), marker=dict(size=4), text=residues))
fig_only.add_trace(go.Scatter(x=[positions[0], positions[-1]], y=[best_thr, best_thr], mode='lines', name=f'Threshold {best_thr:.3f}', line=dict(color='red', width=2, dash='dash')))
fig_only.update_yaxes(range=[0, max(prob_vector.max(), best_thr) * 1.1])
fig_only.update_layout(title=f"LLPS Propensity Profile ({row.uniprot_id})", xaxis_title='Residue Position', yaxis_title='LLPS Propensity', template='simple_white', height=700)

# Figura combinada (tabla + grafico juntos), solo para visualizar en pantalla
combined_fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=False,
    horizontal_spacing=0.1,
    column_widths=[0.3, 0.7],
    specs=[[{"type": "table"}, {"type": "xy"}]]
)
combined_fig.add_trace(fig_only.data[0], row=1, col=2)
combined_fig.add_trace(fig_only.data[1], row=1, col=2)
combined_fig.add_trace(fig_only.data[2], row=1, col=2)
combined_fig.add_trace(go.Table(
    header=dict(values=list(result_df.columns), fill_color='rgba(0,0,0,0)', align='left'),
    cells=dict(values=[result_df[col].map(lambda x: f"{x:.4f}" if isinstance(x, float) else x) for col in result_df.columns], fill_color='rgba(0,0,0,0)', align='left')
), row=1, col=1)
combined_fig.update_layout(
    title=f"LLPS Propensity Profile and Table ({row.uniprot_id})",
    hovermode='x unified',
    height=700,
    width=1500,
    template='simple_white',
    legend=dict(x=0.98, y=0.95, xanchor='right', yanchor='top', bgcolor='rgba(255,255,255,0.8)', bordercolor='lightgray', borderwidth=1)
)
combined_fig.show()

In [ ]:
#@title Download results { display-mode: "form", form-width: "100%" }
from google.colab import files

filename = f"llpsscanai_results_{uniprot_id}"
result_df.to_csv(f"{filename}.csv", index=False)
fig_only.write_html(f"{filename}.html")

files.download(f"{filename}.csv")
files.download(f"{filename}.html")

end_time = time.time()
print(f"Total execution time: {(end_time - start_time)/60:.2f} minutes")